# Train a real custom neural ONNX voice

This notebook builds a **fine-tuned pretrained full LLVC streaming student** using real GPU gradient training, not the prior 65–70 KB Pocket adapter. You supply **one voice recording** that you own or have permission to use. It automatically converts public source speech through a reference-conditioned Seed-VC teacher, builds aligned original/converted pairs, fine-tunes the official pretrained KoeAI LLVC HiFi-GAN architecture, exports ONNX with `audio_in → audio_out`, and tests ONNX Runtime inference.

**Requirements:** Google Colab runtime with an NVIDIA GPU (Runtime → Change runtime type → GPU); a clear voice sample, ideally 15–30 seconds, without music. All generated clips and model files are stored temporarily in the Colab runtime; download the final ONNX. This is **not** on-device training on your Chromebook. Training and model downloads need GPU, storage, and network access. Colab resources and runtime duration are not guaranteed.

**How to use:** The first setup cell creates an isolated Python 3.10 environment, even on Colab hosts running Python 3.13. It does not modify your system Python. If you already tried uploading a recording, the upload cell reuses it after an error. Run the setup cell first if you haven't, then the upload cell, then the other cells in order. The upload cell also works independently. No manual source/target matching or separate RVC training is required. The teacher itself is a pretrained zero-shot converter that uses your sample; the resulting LLVC model is trained to imitate it.

**Verification distinction:** Successful export means a full neural network trained and executes. Only listening tests and target-speaker similarity evaluation establish voice quality. Do not treat an ONNX checksum or filesize as proof that it sounds right.

In [ ]:
# Compatible with Colab hosts that use Python 3.13.
# The models actually run in an isolated Python 3.10 CUDA environment.
import sys
import subprocess
from pathlib import Path

def run(*args, cwd=None):
    command = [str(x) for x in args]
    print("+", " ".join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)

repo = Path("/content/irl-voice")
if (repo / ".git").is_dir():
    run("git", "-C", repo, "pull", "--ff-only")
else:
    run("git", "clone", "--depth", "1",
        "https://github.com/tbenitz/IRL-Voice-Changer.git", repo)

run(sys.executable, repo / "voice-training" / "colab_setup.py")
VOICE_PYTHON = "/content/voice-venv/bin/python"
print("Setup complete. Voice training will use:", VOICE_PYTHON)


## 1. Upload ONE reference voice

Upload a clean WAV, MP3, M4A, or FLAC of the voice to be modeled. The file is sent to your Colab runtime, not to GitHub. Use a recording you have permission to process.

In [ ]:
# Self-contained. It works even if this is the first cell you run,
# and it reuses a file already uploaded to this Colab runtime.
from pathlib import Path
from google.colab import files
import subprocess
import wave

reference = "/content/target_reference.wav"
target = Path(reference)
audio_suffixes = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus"}

if target.is_file():
    print("Using the voice reference already converted:", reference)
else:
    search_dirs = {Path.cwd().resolve(), Path("/content")}
    existing = []
    for folder in search_dirs:
        if folder.is_dir():
            existing.extend(p for p in folder.iterdir()
                            if p.is_file() and p.suffix.lower() in audio_suffixes
                            and p.name != target.name)
    existing = sorted(set(existing), key=lambda p: p.stat().st_mtime, reverse=True)
    if existing:
        source_file = existing[0]
        print("Reusing your already-uploaded audio:", source_file.name)
    else:
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one voice reference recording.")
        source_file = Path(next(iter(uploaded))).resolve()
    if not source_file.exists():
        raise FileNotFoundError(f"Uploaded recording missing: {source_file}")
    subprocess.run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(source_file), "-ac", "1", "-ar", "22050",
        "-t", "30", "-acodec", "pcm_s16le", reference
    ], check=True)

with wave.open(reference, "rb") as voice:
    seconds = voice.getnframes() / voice.getframerate()
    assert voice.getframerate() == 22050 and seconds >= 3, (
        "Voice recording must contain at least 3 seconds of audio.")
    print(f"Voice reference ready: {seconds:.1f}s, {voice.getframerate()} Hz, mono WAV")
print("Your upload is saved. Run the GPU setup cell next if you have not already.")


## 2. Fetch neutral speech and automatically generate real teacher-converted pairs

This uses the LibriSpeech dev-clean corpus (a separate public speech dataset, not your recording) and Seed-VC as an actual reference-conditioned voice-conversion teacher. Unlike two Pocket TTS sentences, each pair is an original recording and a teacher conversion **of that same recording**.

In [ ]:
archive = Path("/content/dev-clean.tar.gz")
if not Path("/content/LibriSpeech/dev-clean").is_dir():
    if not archive.exists():
        run("wget", "-q", "--show-progress", "-O", archive,
            "https://www.openslr.org/resources/12/dev-clean.tar.gz")
    run("tar", "-xzf", archive, "-C", "/content")
# Increase PAIRS to 500 for the larger dataset used in published LLVC workflows.
PAIRS = 300
run("/content/voice-venv/bin/python",
    "/content/irl-voice/voice-training/generate_teacher_pairs.py",
    "--reference", reference,
    "--source-dir", "/content/LibriSpeech/dev-clean",
    "--output", "/content/custom-voice/dataset",
    "--seed-vc", "/content/seed-vc",
    "--pairs", str(PAIRS),
    "--teacher-steps", "10")
print(Path("/content/custom-voice/dataset/manifest.json").read_text()[:1200])

## 3. Train the full LLVC model (not the small Pocket student)

This **fine-tunes the official KoeAI pretrained LLVC HiFi-GAN generator** (not random weights or a 70 KB adapter) using real gradient updates on teacher-converted speech. It saves checkpoints every 500 updates. Higher-quality conversions generally require a substantial paired dataset and extensive GPU training; default values are a starting point, not a guaranteed quality level.

In [ ]:
STEPS = 5000   # Change to 10000+ for a longer training run if GPU session permits.
BATCH = 2
run("/content/voice-venv/bin/python",
    "/content/irl-voice/voice-training/train_full_llvc.py",
    "--llvc", "/content/llvc-upstream",
    "--dataset", "/content/custom-voice/dataset",
    "--output", "/content/custom-voice/llvc-trained",
    "--steps", str(STEPS),
    "--batch", str(BATCH),
    "--name", "my_custom_neural_voice")

## 4. Download and test the actual ONNX

The exporter checks that the ONNX file contains a full trained generator, uses the correct 16 kHz `audio_in` / `audio_out` interface, executes in ONNX Runtime, and produces finite audio at several chunk sizes. Import the ONNX file into **Fast Neural → Import ONNX** on the original IRL Voice Changer site. Listen and benchmark it on your Chromebook. If training or export fails, this cell will not invent or substitute a fake model.

In [ ]:
from google.colab import files
model = Path("/content/custom-voice/llvc-trained/my_custom_neural_voice.onnx")
report = Path("/content/custom-voice/llvc-trained/model_report.json")
assert model.exists() and report.exists(), "No verified trained model. Check the failed cell above."
print(report.read_text())
files.download(str(model))
files.download(str(report))
for ab_name in ("AB_original.wav", "AB_teacher.wav", "AB_trained_ONNX.wav"):
    ab_path = report.parent / ab_name
    assert ab_path.is_file(), "Missing trained ONNX A/B audition clip: " + ab_name
    files.download(str(ab_path))